# FinDirector — Corpus Embedding (GPU)

Embeds the chunk corpus with BGE-M3 on a Colab GPU, then pushes the embedded
file back to HuggingFace for the local pipeline to load into pgvector.

**Before running:** Runtime -> Change runtime type -> L4 GPU (or T4).

Key settings (learned the hard way):
- `max_seq_length = 1024` — caps long tables; ~5% of chunks truncated. Keeps
  attention memory and time bounded.
- `model.half()` — FP16; ~4x faster than FP32 with negligible quality loss.
- `batch_size = 64` with `encode()` sorting inputs by length internally.

In [ ]:
# 1. Dependencies (Colab already has torch + CUDA).
!pip install -q sentence-transformers huggingface_hub

In [ ]:
# 2. Authenticate to HuggingFace (pull corpus, push embedded output).
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Pull the corpus from the HF dataset repo.
from huggingface_hub import hf_hub_download
corpus_path = hf_hub_download(
    repo_id="AlHindi/findirector-corpus", filename="corpus.jsonl", repo_type="dataset",
)
print("corpus at:", corpus_path)

In [ ]:
# 4. Confirm GPU.
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 5. Load BGE-M3 on the GPU, in FP16, capped at 1024 tokens.
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3", device="cuda")
model.max_seq_length = 1024   # cap long tables (truncates ~5% of chunks)
model.half()                  # FP16: ~4x faster than FP32
print("loaded | max_seq_length =", model.max_seq_length, "| fp16")

In [ ]:
# 6. Read chunks.
import json
chunks = [json.loads(l) for l in open(corpus_path) if l.strip()]
print(len(chunks), "chunks to embed")

In [ ]:
# 7. Embed all chunks on GPU -> corpus_embedded.jsonl.
#    encode() sorts by length internally, so batches are memory/time efficient.
import time, json
out_path = "corpus_embedded.jsonl"
texts = [c["text"] for c in chunks]
start = time.time()
vectors = model.encode(
    texts, batch_size=64, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True,
).tolist()
print(f"embedded {len(vectors)} in {time.time()-start:.0f}s")
with open(out_path, "w") as out:
    for c, v in zip(chunks, vectors):
        c["embedding"] = v
        out.write(json.dumps(c) + "\n")
print(f"wrote {out_path}")

In [ ]:
# 8. Sanity-check dimension + normalization.
import json, math
first = json.loads(open(out_path).readline())
emb = first["embedding"]
print("dim:", len(emb), "| L2 norm:", round(math.sqrt(sum(x*x for x in emb)), 4))
assert len(emb) == 1024

In [ ]:
# 9. Push embedded file back to HF for the local loader to pull.
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj=out_path, path_in_repo="corpus_embedded.jsonl",
    repo_id="AlHindi/findirector-corpus", repo_type="dataset",
)
print("uploaded corpus_embedded.jsonl to AlHindi/findirector-corpus")